# Teaching the recitation ear on real voices

This notebook takes the model the app already uses, plays it about a
thousand recordings of ordinary people reciting, and checks whether it comes
out hearing them better than it went in. Nothing here costs money and nothing
is uploaded from your computer: the recordings are pulled straight from the
internet on Google's machine.

## How to run it

1. Go to **colab.research.google.com** and sign in with your Google account.
2. **File > Upload notebook**, and pick this file.
3. **Runtime > Change runtime type > T4 GPU > Save.** This is the free graphics
   card. Without it the training takes days instead of minutes.
4. **Runtime > Run all.** Say yes to the warning that the notebook was not
   written by Google.
5. Leave the tab open. Around twenty five minutes. The last two cells print
   the score before and after, and then offer the trained model as a download.

If the tab is left alone too long Colab disconnects and you start again, so
click into it now and then.

## What it decides

The rule is fixed before the run, so a small wobble cannot be read as a win:
**keep the new model only if it gets at least 1.5 points fewer words wrong**
on recordings it was never trained on. The last cell prints that verdict
itself. If it says no, nothing about the app changes and the run still told
you something worth knowing.

Model: `tarteel-ai/whisper-base-ar-quran` (Apache 2.0), the original of the
one the app runs.
Recordings: `sobolev210/quran-recitation-errors` (MIT) and
`MuazAhmad7/Surah_Ikhlas-Labeled_Dataset` (CC BY 4.0), both public.

In [ ]:
# The graphics card, and the tools. About three minutes.
import torch

if not torch.cuda.is_available():
    raise SystemExit(
        'No graphics card. Runtime > Change runtime type > T4 GPU > Save, then Run all again.'
    )
print('graphics card:', torch.cuda.get_device_name(0))

!pip install -q -U "transformers>=4.44" datasets accelerate peft jiwer librosa soundfile ctranslate2

In [ ]:
# The recordings, and the words they should have said. About five minutes.
import re
import requests
from collections import Counter
from datasets import load_dataset, Audio, Dataset

RATE = 16000

# Plain vowelled spelling, the spelling this model writes. The app's Uthmani
# text scores right words as wrong, so it must not be used as the answer here.
verses = requests.get('https://api.quran.com/api/v4/quran/verses/imlaei', timeout=60).json()['verses']
IMLAEI = {v['verse_key']: v['text_imlaei'] for v in verses}
print(len(IMLAEI), 'ayahs of text')

errors = load_dataset('sobolev210/quran-recitation-errors', split='train').cast_column('audio', Audio(sampling_rate=RATE))
ikhlas = load_dataset('MuazAhmad7/Surah_Ikhlas-Labeled_Dataset', split='train').cast_column('audio', Audio(sampling_rate=RATE))
print(len(errors), 'marked clips,', len(ikhlas), 'al-Ikhlas clips')

In [ ]:
# Only the recitations a reviewer marked nothing on. On a clip where the person
# really did recite wrong, the printed ayah is not what was said, and training
# on it would teach the model to hear a mistake as the right word.
#
# And at most forty clips of any one ayah: two thirds of everything here is
# surah al-Ikhlas, which left alone would teach the model that one surah and
# nothing else.
PER_AYAH = 40


def clean_pairs():
    seen = Counter()
    for row in errors:
        if row.get('riwayah') != 'Hafs':
            continue
        if any(v for v in (row.get('errors') or {}).values()):
            continue
        key = f"{int(row['surah'])}:{int(row['ayah'])}"
        if key in IMLAEI and seen[key] < PER_AYAH:
            seen[key] += 1
            yield {'sound': row['audio']['array'], 'said': IMLAEI[key], 'ayah': key}
    for row in ikhlas:
        if row.get('label') == 0:
            continue
        key = f"112:{row['verse_number']}"
        if key in IMLAEI and seen[key] < PER_AYAH:
            seen[key] += 1
            yield {'sound': row['audio']['array'], 'said': IMLAEI[key], 'ayah': key}


pairs = list(clean_pairs())
print(len(pairs), 'clean recitations,', len({p['ayah'] for p in pairs}), 'different ayahs')

# A fifth held back, split by clip. The model is scored only on recordings it
# never trained on, which is the only score that means anything.
whole = Dataset.from_list(pairs).shuffle(seed=7)
cut = len(whole) // 5
held, taught = whole.select(range(cut)), whole.select(range(cut, len(whole)))
print(len(taught), 'to learn from,', len(held), 'held back to be scored on')

In [ ]:
# How well it hears them today. About four minutes.
import numpy as np
import jiwer
from transformers import WhisperForConditionalGeneration, WhisperProcessor

NAME = 'tarteel-ai/whisper-base-ar-quran'
processor = WhisperProcessor.from_pretrained(NAME, language='ar', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(NAME).cuda()
model.generation_config.language = 'ar'
model.generation_config.task = 'transcribe'
model.generation_config.forced_decoder_ids = None

# Marks off both sides before counting: this measures the letters, and a
# recitation stopped on a word carries a different last vowel by rule, not by
# mistake. Vowels are the sureness score's job, and it is a different model run.
MARKS = re.compile('[\u064b-\u0652\u0670]')
bare = lambda s: MARKS.sub('', s).replace('\u0671', '\u0627').strip()


def wrong_words(m, rows, batch=8):
    heard, meant = [], []
    m.eval()
    for at in range(0, len(rows), batch):
        part = rows[at:at + batch]
        feats = processor(
            [np.asarray(s, dtype=np.float32) for s in part['sound']],
            sampling_rate=RATE, return_tensors='pt',
        ).input_features.to('cuda', dtype=next(m.parameters()).dtype)
        with torch.no_grad():
            out = m.generate(feats, max_new_tokens=180)
        heard += [bare(t) for t in processor.batch_decode(out, skip_special_tokens=True)]
        meant += [bare(t) for t in part['said']]
    return 100 * jiwer.wer(meant, heard), heard


before, said_before = wrong_words(model, held)
print(f'today: {before:.1f}% of words wrong on {len(held)} recordings it has never heard')

In [ ]:
# The teaching itself. About ten minutes on a T4.
#
# LoRA, not a full retrain: a small set of extra numbers alongside the model's
# own, so a thousand recordings cannot wash away everything it learned from
# thousands of hours. A full retrain on this little would do exactly that.
from dataclasses import dataclass
from peft import LoraConfig, get_peft_model
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments


def prepared(row):
    row['input_features'] = processor(
        np.asarray(row['sound'], dtype=np.float32), sampling_rate=RATE,
    ).input_features[0]
    row['labels'] = processor.tokenizer(row['said']).input_ids
    return row


ready = taught.map(prepared, remove_columns=taught.column_names, num_proc=1)


@dataclass
class Gather:
    def __call__(self, rows):
        batch = processor.feature_extractor.pad(
            [{'input_features': r['input_features']} for r in rows], return_tensors='pt')
        marked = processor.tokenizer.pad(
            [{'input_ids': r['labels']} for r in rows], return_tensors='pt')
        # Padding must not be learned as something to say, so it is masked out.
        labels = marked['input_ids'].masked_fill(marked.attention_mask.ne(1), -100)
        if (labels[:, 0] == processor.tokenizer.bos_token_id).all():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch


tuned = get_peft_model(model, LoraConfig(
    r=32, lora_alpha=64, lora_dropout=0.05, bias='none',
    target_modules=['q_proj', 'v_proj'],
))
tuned.print_trainable_parameters()

Seq2SeqTrainer(
    model=tuned,
    args=Seq2SeqTrainingArguments(
        output_dir='ear-lora',
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        learning_rate=1e-3,
        warmup_steps=40,
        num_train_epochs=4,
        fp16=True,
        logging_steps=25,
        save_strategy='no',
        remove_unused_columns=False,
        label_names=['labels'],
        report_to=[],
    ),
    train_dataset=ready,
    data_collator=Gather(),
).train()

In [ ]:
# The same held-back recordings, scored again. About four minutes.
BAR = 1.5

after, said_after = wrong_words(tuned, held)
print(f'today:  {before:.1f}% of words wrong')
print(f'taught: {after:.1f}% of words wrong')
print(f'change: {before - after:+.1f} points on {len(held)} recordings it never trained on')
print()
won = (before - after) >= BAR
print('VERDICT: keep it, and score it against the 1,950 clips at home.' if won
      else f'VERDICT: not worth keeping. The bar was {BAR} points and it did not clear it.')
print()
print('Five it still gets wrong:')
shown = 0
for meant, heard in zip(held['said'], said_after):
    if bare(meant) != heard and shown < 5:
        shown += 1
        print(f'  should be: {meant}')
        print(f'  heard:     {heard}')

In [ ]:
# The model, in the shape the app loads. About three minutes.
#
# The extra numbers are folded back into the model, then converted the way the
# one the app runs today was converted, so nothing at home has to change except
# which folder recitation_model points at.
if not won:
    print('It did not clear the bar, so there is nothing to bring home. Nothing was downloaded.')
else:
    merged = tuned.merge_and_unload()
    merged.save_pretrained('ear-merged')
    processor.save_pretrained('ear-merged')
    !ct2-transformers-converter --model ear-merged --output_dir faster-whisper-base-ar-quran-tuned --copy_files preprocessor_config.json --quantization float16
    !zip -qr ear-tuned.zip faster-whisper-base-ar-quran-tuned
    from google.colab import files
    print('Saving to your computer. Put the unzipped folder in the project and point')
    print('recitation_model in backend/config.py at it.')
    files.download('ear-tuned.zip')